In [33]:
#importing everything needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

#Setting up some additional display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [4]:
# calling to dataset
df = pd.read_csv("data_cleaned (2).csv")

print("Shape:", df.shape)
display(df.head())
display(df.info())

Shape: (4872, 19)


,Country Name,Year,Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Agriculture, forestry, and fishing, value added (% of GDP)",carbon_emissions,Electric power consumption (kWh per capita),Electricity production from coal sources (% of total),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from nuclear sources (% of total),"Electricity production from oil, gas and coal sources (% of total)","Energy imports, net (% of energy use)",Fossil fuel energy consumption (% of total),GDP (constant LCU),GDP (current LCU),"Net trade in goods and services (BoP, current US$)",Urban population,Urban population (% of total population)
0,Afghanistan,2001,9.3000,NaN,NaN,0.9402,NaN,0.0000,93.5373,0.0000,0.0000,6.4627,NaN,NaN,"378,788,779,600.0000","133,474,620,100.0000",NaN,"3,801,530.0000",18.7412
1,Afghanistan,2002,14.1000,NaN,38.6279,0.9388,NaN,0.0000,93.5374,0.0000,0.0000,6.4626,NaN,NaN,"487,122,375,000.0000","182,162,721,300.0000",NaN,"4,049,282.0000",18.9412
2,Afghanistan,2003,19.0000,2.1000,37.4189,1.0172,NaN,0.0000,80.6309,0.0000,0.0000,19.3691,NaN,NaN,"530,146,376,400.0000","221,358,563,000.0000",NaN,"4,355,660.0000",19.1600
3,Afghanistan,2004,23.8000,7.8000,29.7211,0.9024,NaN,0.0000,62.8649,0.0000,0.0000,37.1351,NaN,NaN,"537,643,271,700.0000","249,791,940,700.0000",NaN,"4,570,616.0000",19.3994
4,Afghanistan,2005,28.7000,15.4000,31.1149,1.2763,NaN,0.0000,63.0140,0.0000,0.0000,36.9860,NaN,NaN,"598,019,077,900.0000","308,163,225,800.0000",NaN,"4,809,142.0000",19.7059


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4872 entries, 0 to 4871
Data columns (total 19 columns):
 #   Column                                                              Non-Null Count  Dtype  
---  ------                                                              --------------  -----  
 0   Country Name                                                        4872 non-null   object 
 1   Year                                                                4872 non-null   int64  
 2   Access to electricity (% of population)                             4631 non-null   float64
 3   Access to electricity, rural (% of rural population)                4393 non-null   float64
 4   Agriculture, forestry, and fishing, value added (% of GDP)          4478 non-null   float64
 5   carbon_emissions                                                    4872 non-null   float64
 6   Electric power consumption (kWh per capita)                         3375 non-null   float64
 7   Electricity pro

None

In [6]:
#Defining my target variable and my features
target = "carbon_emissions"

X = df.drop(columns=[target])
y = df[target]

print("Feature columns:")
print(X.columns.tolist())
print("\nTarget column:")
print(target)

Feature columns:
['Country Name', 'Year', 'Access to electricity (% of population)', 'Access to electricity, rural (% of rural population)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from coal sources (% of total)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Electricity production from oil, gas and coal sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'GDP (constant LCU)', 'GDP (current LCU)', 'Net trade in goods and services (BoP, current US$)', 'Urban population', 'Urban population (% of total population)']

Target column:
carbon_emissions


In [8]:
#identifyng which ones of my columns are numerical and categorical
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: ['Country Name']
Numeric features: ['Year', 'Access to electricity (% of population)', 'Access to electricity, rural (% of rural population)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from coal sources (% of total)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Electricity production from oil, gas and coal sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'GDP (constant LCU)', 'GDP (current LCU)', 'Net trade in goods and services (BoP, current US$)', 'Urban population', 'Urban population (% of total population)']


In [10]:
# creating train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (3897, 18)
Testing shape: (975, 18)


In [35]:
# Did some preprocessing for missing values and changing 
# Numeric:
# filling missing values with median
# scaling some of the features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical:
# one hot encode Country Name
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [37]:
# created a helper function to evalute the models we choose
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    results = {
        "Model": name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred)
    }

    return results

In [39]:
# Linear Regression Baseline
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_results = evaluate_model(
    "Linear Regression",
    linear_model,
    X_train, X_test, y_train, y_test
)

linear_results

{'Model': 'Linear Regression',
 'Train R2': 0.9904974393975519,
 'Test R2': 0.980709954367265,
 'Train RMSE': 82.04079158632224,
 'Test RMSE': 90.34721684207665,
 'Train MAE': 33.12711072585302,
 'Test MAE': 34.487272584587906}

In [ ]:
#Based on the results we can see that the R2 for both train and test are not only extremely good but also very similar.
#This may be caused by on hot coding countries, which might be improving the overall model. The closeness between both 
#the training and testing does suggest limited overfitting which is good for the model. 

In [41]:
# Ridge Regression
ridge_alphas = np.logspace(-3, 3, 50)

ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RidgeCV(alphas=ridge_alphas, cv=5))
])

ridge_results = evaluate_model(
    "Ridge Regression",
    ridge_model,
    X_train, X_test, y_train, y_test
)

ridge_results

{'Model': 'Ridge Regression',
 'Train R2': 0.9904659760941422,
 'Test R2': 0.9806716883367695,
 'Train RMSE': 82.17649928047086,
 'Test RMSE': 90.43678419137753,
 'Train MAE': 33.672696230248754,
 'Test MAE': 35.03341579824433}

In [43]:
# Finding the best alpha for Ridge
best_ridge_alpha = ridge_model.named_steps["regressor"].alpha_
print("Best Ridge alpha:", best_ridge_alpha)

Best Ridge alpha: 0.0030888435964774815


In [ ]:
#Ridge Regression created nearly identical results to the baseline model. Since the optimal regularization 
#strength was very small, the Ridge model did not really improve performance, meaning that coefficient shrinkage
#was not really necessary for this dataset.

In [25]:
#Lasso Regression
lasso_alphas = np.logspace(-3, 1, 50)

lasso_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LassoCV(alphas=lasso_alphas, cv=5, random_state=42, max_iter=20000))
])

lasso_results = evaluate_model(
    "Lasso Regression",
    lasso_model,
    X_train, X_test, y_train, y_test
)

lasso_results

{'Model': 'Lasso Regression',
 'Train R2': 0.9904968660811188,
 'Test R2': 0.9806957559992278,
 'Train RMSE': 82.04326642581783,
 'Test RMSE': 90.3804605963986,
 'Train MAE': 33.12083318856221,
 'Test MAE': 34.48003897281203}

In [27]:
# Best alpha for Lasso
best_lasso_alpha = lasso_model.named_steps["regressor"].alpha_
print("Best Lasso alpha:", best_lasso_alpha)

Best Lasso alpha: 0.001


In [ ]:
#Lasso Regression also performed nearly identically to the baseline. While it is useful for feature selection, 
#it still keeping a big number of predictors, especially after one hot encoding country names. This is telling us 
#that many country specific and numeric variables contribute to this predictive performance.

In [29]:
# =========================================
# 11. Compare results
# =========================================
results_df = pd.DataFrame([linear_results, ridge_results, lasso_results])
display(results_df)

,Model,Train R2,Test R2,Train RMSE,Test RMSE,Train MAE,Test MAE
0,Linear Regression,0.9905,0.9807,82.0408,90.3472,33.1271,34.4873
1,Ridge Regression,0.9905,0.9807,82.1765,90.4368,33.6727,35.0334
2,Lasso Regression,0.9905,0.9807,82.0433,90.3805,33.1208,34.4800


In [ ]:
#Overall, all three models performed veyr closely to each other, with no major improvement from regularization. 
#This tells us that the dataset already supports a strong linear fit, but some of the predictive 
#power may come from country level identity effects rather than only from the economical development indicators.

In [31]:
# Showing Lasso selecyted features
# Get transformed feature names
feature_names = lasso_model.named_steps["preprocessor"].get_feature_names_out()

# Get Lasso coefficients
lasso_coefficients = lasso_model.named_steps["regressor"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lasso_coefficients
})

# Features kept by Lasso
selected_features = coef_df[coef_df["Coefficient"] != 0].sort_values(
    by="Coefficient", key=np.abs, ascending=False
)

print("Number of features selected by Lasso:", selected_features.shape[0])
display(selected_features)

Number of features selected by Lasso: 219


,Feature,Coefficient
103,cat__Country Name_India,"-4,198.1562"
210,cat__Country Name_United States,"2,461.3577"
56,cat__Country Name_China,"-2,065.4624"
42,cat__Country Name_Brazil,"-2,023.1341"
104,cat__Country Name_Indonesia,"-1,479.1687"
...,...,...
8,num__Electricity production from nuclear sourc...,-2.2501
145,cat__Country Name_Namibia,1.2280
40,cat__Country Name_Bosnia and Herzegovina,-0.7076
77,cat__Country Name_Eswatini,-0.4949
